# AutoPBR v2 — Run Pipeline (config-driven)

Questo notebook esegue la pipeline leggendo **un unico file di configurazione**:

- `config/pipeline.yaml`

Vantaggi:
- un solo posto dove cambiare parametri (SAM3 / Nemotron / soglie / paths)
- riproducibile per colleghi e reviewer
- stesso comportamento del terminale (esecuzione via CLI)

> Prima di eseguire: installa `pyyaml` (`pip install pyyaml`) e imposta `NVIDIA_API_KEY`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shlex
import yaml

CONFIG_PATH = Path("config/pipeline.yaml")
REPO_ROOT = Path(".").resolve()

print("Repo:", REPO_ROOT)
print("Config:", CONFIG_PATH.resolve())


## Helper: run comandi con output live


In [ ]:
def run(cmd: str):
    print("\n▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def q(x) -> str:
    return shlex.quote(str(x))

def args_to_cli(args: dict, ctx: dict) -> str:
    parts = []
    for k, v in args.items():
        flag = f"--{k}"
        if isinstance(v, bool):
            if v:
                parts.append(flag)
            continue
        if isinstance(v, str):
            v = v.format(**ctx)
        parts.append(flag)
        parts.append(str(v))
    return " ".join(q(p) for p in parts)


## Load config + sanity checks


In [ ]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing {CONFIG_PATH}. Create/commit it in the repo.")

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

photos_dir = cfg["paths"]["photos_dir"]
out_root   = cfg["paths"]["output_root"]
ctx = {"photos_dir": photos_dir, "output_root": out_root}

# Env check
missing = [k for k in cfg.get("env", {}).get("required", []) if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}. Set them before running (e.g., export NVIDIA_API_KEY=...).")
print("✅ Env OK")

# Input folder check
photos_path = REPO_ROOT / photos_dir
photos_path.mkdir(exist_ok=True, parents=True)
imgs = list(photos_path.glob("*.jpg")) + list(photos_path.glob("*.jpeg")) + list(photos_path.glob("*.png"))
print(f"📷 images in {photos_path}: {len(imgs)}")


## Step 1 — SAM3 (segmentation)

Nota: i prompt nel YAML sono documentativi; se `2_sam3hi.py` non espone flag per i prompt, li ignora comunque.
Qui passiamo solo flag stabili (paths / per_building / bbox gating) *se presenti nel YAML*.


In [ ]:
sam3 = cfg.get("sam3", {})
if sam3.get("enabled", True):
    script = sam3.get("script", "2_sam3hi.py")

    # pass only known scalar args if provided
    args = {}
    for k in ["photos_dir", "out_dir", "per_building", "min_inst_bbox_h_wall", "min_inst_bbox_h_roof"]:
        if k in sam3:
            args[k] = sam3[k]

    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ SAM3 disabled in config")


## Step 2 — Proxy semantic check


In [ ]:
proxy = cfg.get("proxy_check", {})
if proxy.get("enabled", True):
    script = proxy.get("script", "3_proxy_semantic_check.py")
    args = {k: v for k, v in proxy.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ proxy_check disabled in config")


## Step 3 — Nemotron structured (ECCV frozen config)


In [ ]:
ns = cfg.get("nemotron_structured", {})
if ns.get("enabled", True):
    script = ns.get("script", "4_nemotron_building_priors.py")
    args = {k: v for k, v in ns.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ nemotron_structured disabled in config")


## Step 4 — Baseline full-image (optional)


In [ ]:
nbaseline = cfg.get("nemotron_baseline", {})
if nbaseline.get("enabled", False):
    script = nbaseline.get("script", "5_nemotron_fullimage.py")
    args = {k: v for k, v in nbaseline.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ baseline disabled in config")


## Step 5 — Validation (optional)


In [ ]:
val = cfg.get("validation", {})
if val.get("enabled", False):
    script = val.get("script", "validator.py")
    args = {k: v for k, v in val.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ validation disabled in config")


## Quick check outputs


In [ ]:
out_candidates = [
    REPO_ROOT / "materials_v2_filtered.json",
    REPO_ROOT / "baseline_full_image.json",
    REPO_ROOT / out_root / "sam3_instances" / "manifest.json",
    REPO_ROOT / out_root / "validation_results" / "report.csv",
]
for p in out_candidates:
    print(("✅" if p.exists() else "❌"), p)
